# WP39 — Web API & Real-Time Monitoring Dashboard (v0.4)
## PrometheusAPI · APIMetrics · LiveEventStream · DashboardRenderer

Demonstrates **WP39**: a pure-stdlib REST API server with live event streaming and an auto-refreshing HTML monitoring dashboard — no Flask, FastAPI, or external dependencies required.

Runtime: **~2 min**

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC'); sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')); 
    if repo_root not in sys.path: sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import time, random, numpy as np, matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp39_web_api import (
    APIMetrics, LiveEventStream, PrometheusAPI, DashboardRenderer,
    PrometheusAPIServer, verify_wp39_exit_criteria
)
print('WP39 imports OK')

In [ ]:
# APIMetrics — ring-buffer metrics store
m = APIMetrics(max_history=500)
import time, math

# Simulate recording metrics over 20 'generations'
for gen in range(20):
    acc = 0.60 + gen * 0.015 + (hash(gen) % 7 - 3) * 0.005
    m.record('accuracy', acc)
    m.record('generation', float(gen))
    m.record('n_strategies', 6.0)
    time.sleep(0.01)  # small delay for realistic timestamps

snap = m.snapshot()
print('APIMetrics snapshot:')
print(f'  uptime_s:   {snap["uptime_s"]:.2f}s')
print(f'  metrics:    {snap["metrics"]}')
print()
hist = m.get_history('accuracy', last_n=5)
print(f'Accuracy history (last 5): {[round(v,3) for _,v in hist]}')

In [ ]:
# LiveEventStream
stream = LiveEventStream()
received = []
stream.subscribe(lambda name, payload: received.append((name, payload)))

stream.publish('generation_complete', {'gen': 1, 'acc': 0.72})
stream.publish('proof_found', {'theorem': 'add_zero', 'depth': 1})
stream.publish('elo_updated', {'agent': 'prometheus_v0', 'mu': 1543.2})

recent = stream.recent_events(10)
print(f'Published {len(received)} events, received by subscriber:')
for name, payload in received:
    print(f'  [{name}]  {payload}')

In [ ]:
# PrometheusAPI — route handling
api = PrometheusAPI(metrics=m, events=stream)

# Test each endpoint
tests = [
    ('GET', '/health', {}),
    ('GET', '/metrics', {}),
    ('GET', '/metrics/elo', {'domain': ['synthesis']}),
    ('GET', '/events', {}),
    ('POST', '/synthesise', {}, b'{"strategy": "promote_best"}'),
    ('POST', '/prove',      {}, b'{"theorem": "add_zero"}'),
    ('POST', '/evolve',     {}, b'{"generations": 3}'),
    ('GET', '/notfound', {}),
]
print(f'  {"Method":<5} {"Path":<22} {"Status":<6} {"Content preview"}')
print('-' * 80)
for t in tests:
    method, path = t[0], t[1]
    query  = t[2] if len(t) > 2 else {}
    body   = t[3] if len(t) > 3 else None
    status, ctype, resp = api.handle_request(method, path, query, body)
    preview = resp[:60].replace('\n','')
    print(f'  {method:<5} {path:<22} {status:<6} {preview}')

In [ ]:
# DashboardRenderer — HTML page
renderer = DashboardRenderer()
html = renderer.render(m.snapshot(), stream.recent_events(20))
print(f'HTML length: {len(html)} chars')
print(f'Contains <!DOCTYPE html>: {"<!DOCTYPE html>" in html}')
print(f'Contains metrics table:   {"Current Metrics" in html}')
print(f'Contains events table:    {"Recent Events" in html}')
print()
print('--- HTML snippet (first 500 chars) ---')
print(html[:500])
print('...')

In [ ]:
# Background server demonstration (start + verify + stop)
server = PrometheusAPIServer(api, host='127.0.0.1', port=0)  # port=0 -> OS assigns
# Note: can't use port=0 with stdlib HTTPServer directly; use a fixed port
server2 = PrometheusAPIServer(api, host='127.0.0.1', port=18765)
server2.start()
print(f'Server running: {server2.is_running()}')
print(f'Base URL: {server2.base_url}')
import urllib.request, json
try:
    with urllib.request.urlopen(f'{server2.base_url}/health', timeout=2) as resp:
        data = json.loads(resp.read())
        print(f'HTTP GET /health -> status={data["status"]}  version={data["version"]}')
except Exception as e:
    print(f'HTTP test skipped: {e}')
finally:
    server2.stop()
    print(f'Server stopped: {not server2.is_running()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
acc_hist = m.get_history('accuracy')
ts   = [t for t,_ in acc_hist]
accs = [v for _,v in acc_hist]
if ts:
    ts_rel = [t - ts[0] for t in ts]
    ax.plot(ts_rel, accs, 'o-', color='#4CAF50', markersize=4)
ax.set_xlabel('Time elapsed (s)'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Metric — Ring Buffer History', fontweight='bold')

ax2 = axes[1]
event_counts = {}
for ev in stream.recent_events(50):
    event_counts[ev['event']] = event_counts.get(ev['event'], 0) + 1
if event_counts:
    ax2.bar(event_counts.keys(), event_counts.values(), color='#2196F3', edgecolor='black', alpha=0.85)
ax2.set_ylabel('Count'); ax2.set_title('Event Stream — Published Events', fontweight='bold')
import matplotlib.ticker as mticker
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.suptitle('WP39: Web API & Real-Time Monitoring Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp39_web_api.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved wp39_web_api.png')

In [ ]:
criteria = verify_wp39_exit_criteria(api, server)
print('WP39 Exit Criteria Verification'); print('='*60)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()): print('\nAll WP39 exit criteria satisfied.')

---
## Conclusions

**WP39** provides external observability of the Prometheus stack:
- Pure-stdlib HTTP server (no Flask/FastAPI dependency)
- `APIMetrics` ring-buffer survives high-frequency recording
- `LiveEventStream` fan-out publish-subscribe pattern
- `DashboardRenderer` auto-refresh HTML — no JS framework

### Usage in production
```python
from prometheus.wp39_web_api import PrometheusAPI, PrometheusAPIServer
api = PrometheusAPI(); server = PrometheusAPIServer(api, port=8765)
server.start()  # serves at http://localhost:8765
```

### References
- Kleppmann (2017) *Designing Data-Intensive Applications*
- Sculley et al. (2015) *Hidden Technical Debt in Machine Learning Systems*
- Good (1965) — transparency as a prerequisite for ultraintelligent systems